In [7]:
import os
import json
import time
from typing import Optional, List
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool, ToolException
from pydantic import BaseModel, Field, field_validator

In [8]:
# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("GROQ_API_KEY not set")
# 初始化模型
model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [9]:
def example1():
    # 高级：使用 Pydantic 参数模型 定义 工具
    class WeatherInput(BaseModel):
        """天气查询参数"""

        city: str = Field(description="城市名称")
        unit: str = Field(description="温度单位：celsius 或 fahrenheit", default="celsius")

        @field_validator("unit")
        @classmethod
        def validate_unit(cls, v):
            if v not in ["celsius", "fahrenheit"]:
                raise ValueError("unit 必须是 'celsius' 或 'fahrenheit'")
            return v
    @tool(args_schema=WeatherInput)
    def get_weather(city: str, unit: str = "celsius") -> str:
        """获取指定城市的天气信息"""
        # 模拟天气数据
        weather_data = {
            "北京": {"temp_c": 15, "condition": "晴"},
            "上海": {"temp_c": 20, "condition": "多云"},
            "深圳": {"temp_c": 25, "condition": "阴"},
        }
        data = weather_data.get(city, {"temp_c": 18, "condition": "未知"})
        temp = data["temp_c"] if unit == "celsius" else data["temp_c"] * 9/5 + 32
        unit_symbol = "℃" if unit == "celsius" else "℉"
        return f"{city} 的天气是 {data['condition']}，温度是 {temp:.1f}{unit_symbol}"
    # 工具使用示例
    print("\n 测试天气查询工具")
    print(get_weather.invoke({"city": "北京", "unit": "celsius"}))
    print(get_weather.invoke({"city": "上海", "unit": "fahrenheit"}))

    # 工具的错误处理机制
    @tool
    def safe_divide(a: float, b: float) -> str:
        """安全的除法运算。

        Args:
            a: 被除数
            b: 除数
        """
        try:
            if b == 0:
                raise ToolException("除数不能为零！请提供一个非零的除数。")
            return f"{a} ÷ {b} = {a / b:.4f}"
        except ToolException as e:
            return str(e)
    # 自定义错误处理
    def custom_error_handler(error: ToolException) -> str:
        return f"⚠️ 操作失败: {error.args[0]}\n💡 建议: 请检查输入参数是否正确。"
    # 自定义错误处理
    @tool
    def fetch_data(url: str) -> str:
        """从 URL 获取数据（模拟）。

        Args:
            url: 要获取数据的 URL
        """
        try:
            if not url.startswith("http"):
                raise ToolException("URL 必须以 http:// 或 https:// 开头")
            if "error" in url:
                raise ToolException("无法连接到服务器")
            return f"成功获取数据: {url}"
        except ToolException as e:
            return f"⚠️ 操作失败: {e}\n💡 建议: 请检查输入参数是否正确。"
    # 带重试的工具
    @tool
    def unreliable_api(query: str) -> str:
        """模拟不稳定的 API 调用。"""
        import random
        if random.random() < 0.3:
            raise Exception("API 临时不可用")
        return f"API 返回: {query}"

    def with_retry(tool_func, max_retries: int = 3):
        """添加重试逻辑的包装器"""
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(max_retries):
                try:
                    return tool_func.invoke(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  重试 {attempt + 1}/{max_retries}: {e}")
                    time.sleep(0.1)
            return f"失败（{max_retries}次重试后）: {last_error}"
        return wrapper

    def with_retry(tool_func, max_retries: int = 3):
        """添加重试逻辑的包装器"""
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(max_retries):
                try:
                    return tool_func.invoke(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  重试 {attempt + 1}/{max_retries}: {e}")
                    time.sleep(0.1)
            return f"失败（{max_retries}次重试后）: {last_error}"
        return wrapper
    print("\n工具的错误处理机制")
    print("\n测试安全除法:")
    print(safe_divide.invoke({"a": 10.0, "b": 0.0}))  # 触发错误
    print("\n测试自定义错误处理:")
    print(fetch_data.invoke({"url": "invalid-url"}))  # 触发错误
    print("\n测试带重试的 API:")
    retry_api = with_retry(unreliable_api)
    print(retry_api({"query": "测试查询"}))


In [11]:
def example2():
    """
    演示完整的生产级智能体
    注意：此示例需要配置有效的 API Key
    """
    print("\n" + "=" * 60)
    print("示例 5: 完整的智能体示例")
    print("=" * 60)

    # 检查 API Key
    api_key = os.getenv("DEEPSEEK_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("\n⚠️ 未检测到 API Key，跳过此示例")
        print("请设置 DEEPSEEK_API_KEY 或 OPENAI_API_KEY 环境变量")
        return

    from langchain.chat_models import init_chat_model
    from langchain.agents import create_agent
    from langgraph.checkpoint.memory import MemorySaver
    from langchain_core.tools import tool
    from langchain_core.callbacks import BaseCallbackHandler
    from pydantic import BaseModel, Field

    # ====== 定义工具 ======

    class OrderInput(BaseModel):
        order_id: str = Field(description="订单号，格式: ORD+数字")

    # 模拟数据库
    ORDERS = {
        "ORD001": {"status": "已发货", "product": "iPhone 15", "amount": 7999.0},
        "ORD002": {"status": "处理中", "product": "MacBook Pro", "amount": 14999.0},
        "ORD003": {"status": "已完成", "product": "AirPods Pro", "amount": 1899.0},
    }

    @tool(args_schema=OrderInput)
    def check_order(order_id: str) -> str:
        """查询订单状态。"""
        order = ORDERS.get(order_id.upper())
        if not order:
            return f"未找到订单 {order_id}。可用订单: {', '.join(ORDERS.keys())}"
        return json.dumps({
            "订单号": order_id.upper(),
            "商品": order["product"],
            "状态": order["status"],
            "金额": f"¥{order['amount']}"
        }, ensure_ascii=False, indent=2)

    @tool
    def get_faq(topic: str) -> str:
        """获取常见问题解答。

        Args:
            topic: 问题主题，如"配送"、"退货"、"支付"
        """
        faqs = {
            "配送": "普通快递3-5天送达，顺丰1-2天送达，偏远地区可能延迟1-2天",
            "退货": "支持7天无理由退货，商品需保持原包装完好",
            "支付": "支持微信、支付宝、银行卡等多种支付方式",
            "发票": "订单完成后可在'我的订单'中申请电子发票"
        }

        for key, value in faqs.items():
            if key in topic:
                return f"📖 关于【{key}】:\n{value}"

        return f"暂无关于'{topic}'的FAQ，常见主题: {', '.join(faqs.keys())}"

    @tool
    def calculate_shipping(address: str) -> str:
        """计算配送费用。

        Args:
            address: 收货地址
        """
        # 简单的运费计算逻辑
        if any(city in address for city in ["北京", "上海", "广州", "深圳"]):
            return f"配送到 {address}: 免运费（一线城市包邮）"
        elif any(region in address for region in ["新疆", "西藏", "内蒙古"]):
            return f"配送到 {address}: ¥25（偏远地区）"
        else:
            return f"配送到 {address}: ¥10（标准运费）"

    # ====== 监控回调 ======

    class AgentMonitor(BaseCallbackHandler):
        def on_tool_start(self, serialized, input_str, **kwargs):
            print(f"  🔧 调用: {serialized.get('name', '?')}")

        def on_tool_end(self, output, **kwargs):
            print(f"  ✅ 返回: {str(output)[:80]}...")

    # ====== 创建智能体 ======

    # 确定使用哪个模型
    if os.getenv("DEEPSEEK_API_KEY"):
        model = init_chat_model("deepseek-chat", model_provider="deepseek")
    else:
        model = init_chat_model("gpt-3.5-turbo", model_provider="openai")

    tools = [check_order, get_faq, calculate_shipping]
    memory = MemorySaver()

    system_prompt = """你是一个专业的电商客服助手。

你可以：
1. 查询订单状态（可用订单号: ORD001, ORD002, ORD003）
2. 回答常见问题（配送、退货、支付、发票）
3. 计算配送费用

请保持友好、专业的态度，回答要简洁明了。"""

    agent = create_agent(
        model=model,
        tools=tools,
        checkpointer=memory,
        system_prompt=system_prompt
    )

    # ====== 测试对话 ======

    config = {
        "configurable": {"thread_id": "demo_session"},
        "callbacks": [AgentMonitor()]
    }

    test_queries = [
        "帮我查一下订单 ORD001",
        "配送一般要多久？",
        "我在杭州，运费是多少？"
    ]

    print("\n📞 开始客服对话模拟:")
    print("-" * 40)

    for query in test_queries:
        print(f"\n👤 用户: {query}")

        try:
            response = agent.invoke(
                {"messages": [{"role": "user", "content": query}]},
                config=config
            )
            ai_message = response["messages"][-1].content
            print(f"🤖 客服: {ai_message}")
        except Exception as e:
            print(f"❌ 错误: {e}")

        print("-" * 40)

# ============================================================
# 主函数
# ============================================================

In [12]:
def main():
    print("\n"+"="*70)
    print("tools_and_agent")
    print("="*70)

    example1()
    print("\n"+"="*70)
    example2()

In [13]:
if __name__ == "__main__":
    main()


tools_and_agent

 测试天气查询工具
北京 的天气是 晴，温度是 15.0℃
上海 的天气是 多云，温度是 68.0℉

工具的错误处理机制

测试安全除法:
除数不能为零！请提供一个非零的除数。

测试自定义错误处理:
⚠️ 操作失败: URL 必须以 http:// 或 https:// 开头
💡 建议: 请检查输入参数是否正确。

测试带重试的 API:
  重试 1/3: API 临时不可用
API 返回: 测试查询


示例 5: 完整的智能体示例

📞 开始客服对话模拟:
----------------------------------------

👤 用户: 帮我查一下订单 ORD001
  🔧 调用: check_order
  ✅ 返回: content='{\n  "订单号": "ORD001",\n  "商品": "iPhone 15",\n  "状态": "已发货",\n  "金额": "¥...
🤖 客服: 根据查询结果，您的订单 ORD001 状态如下：

**订单号：** ORD001
**商品：** iPhone 15
**状态：** 已发货
**金额：** ¥7999.0

您的订单已经发货了！如果您需要了解具体的物流信息或有其他问题，请随时告诉我。
----------------------------------------

👤 用户: 配送一般要多久？
  🔧 调用: get_faq
  ✅ 返回: content='📖 关于【配送】:\n普通快递3-5天送达，顺丰1-2天送达，偏远地区可能延迟1-2天' name='get_faq' tool_call_i...
🤖 客服: 根据我们的配送政策：

**配送时效：**
- **普通快递：** 3-5天送达
- **顺丰快递：** 1-2天送达
- **偏远地区：** 可能延迟1-2天

您的订单 ORD001 已经发货，具体配送时间取决于您选择的快递方式和收货地址。如果您想了解更准确的预计送达时间，可以告诉我您的收货地址，我可以帮您计算更具体的配送情况。
----------------------------------------

👤 用户: 我在杭州，运费是多少？
  🔧 调用: calculate_shipping
  ✅ 返回: con